# 97 — Kiểm GPT-QModel với `gpt_neox` trên Pythia-70M (việc của TV-A, hạn 18/11)

Câu hỏi cần trả lời trước khi viết SPEC C1 (GPTQ 4 bit):

1. GPT-QModel có lượng tử hoá được Pythia (`gpt_neox`) trên Kaggle T4 với torch/transformers sẵn có không?
2. Có lấy lại được trọng số **giải lượng tử (dequantize) ở fp32** để chạy theo thiết kế giả lập (D-04) và nút vặn W(α) không?
3. Sai số trọng số và perplexity của GPTQ so với RTN cùng cấu hình (bit, group, bất đối xứng) ra sao?

Cách chạy: **từng ô (QT-11, trường hợp b: lỗi môi trường cần thử nhiều lần)**, GPU T4, bật Internet.
Chạy xong thì **Stop session**. Ước lượng dưới 15 phút.

Claude đã chạy `TOY_TEST = True` trên CPU (model GPT-NeoX ngẫu nhiên cỡ Pythia-70M, GPT-QModel 7.6.0,
transformers 5.22.2, torch 2.15). Kết quả CPU không thay được lần chạy thật trên Kaggle.

In [1]:
# Tham số (papermill). Comment không có dấu bằng (QT-21).
MODEL_ID = "EleutherAI/pythia-70m"
REVISION = "main"
# Bản 7.7.0 trở lên đòi torch từ 2.14.1, sẽ nâng torch của Kaggle; 7.6.0 đòi torch từ 2.8.0
GPTQMODEL_VERSION = "7.6.0"
INSTALL = True
BITS = 4
GROUP_SIZE = 128
# Khớp RTN của G0b: bất đối xứng
SYM = False
DESC_ACT = False
# GPT-QModel 7.6.0 mặc định bật act_group_aware cho GPTQ; tắt để có GPTQ chuẩn
ACT_GROUP_AWARE = False
DAMP_PERCENT = 0.05
N_CALIB = 128
CALIB_LEN = 512
N_PPL = 40
PPL_LEN = 512
OUT_DIR = "outputs/97_kiem_gptqmodel"
TOY_TEST = False
# Chỉ dùng khi TOY_TEST: đường dẫn file tokenizer.json của GPT-NeoX-20B
TOY_TOKENIZER_JSON = ""

In [2]:
# Parameters
TOY_TEST = True
INSTALL = False
TOY_TOKENIZER_JSON = "/home/claude/ext/tok/BlinkDL_RWKV-LM_main_RWKV-v4_20B_tokenizer.json"
OUT_DIR = "/home/claude/work/runs/out97"
N_CALIB = 32
N_PPL = 8
CALIB_LEN = 128
PPL_LEN = 128


## Ô 1. Cài GPT-QModel đúng bản và in phiên bản (QT-26)

Ghi lại phiên bản torch **trước và sau** khi cài. Nếu torch bị đổi, dừng và báo lại (không chạy tiếp các ô sau).
`GPTQMODEL_CPU_WORKERS` đặt bằng 2 để tránh lỗi pool luồng trên máy ít nhân (gặp khi Claude chạy thử).

In [3]:
import os, sys, json, time, subprocess, importlib
os.environ.setdefault("GPTQMODEL_CPU_WORKERS", "2")
import torch
torch_before = torch.__version__
if INSTALL and not TOY_TEST:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"gptqmodel=={GPTQMODEL_VERSION}"], check=True)
# gptqmodel in log ra stdout khi import, nên chỉ đọc dòng có dấu VER
_o = subprocess.run([sys.executable, "-c", "import json, torch, transformers, numpy, gptqmodel; print('VER', json.dumps([torch.__version__, transformers.__version__, numpy.__version__, gptqmodel.__version__]))"],
                    capture_output=True, text=True).stdout
out = json.loads([l for l in _o.splitlines() if l.startswith("VER ")][-1][4:])
INFO = {"torch_truoc": torch_before, "torch_sau": out[0], "transformers": out[1], "numpy": out[2], "gptqmodel": out[3],
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}
print(INFO)
if INFO["torch_sau"] != INFO["torch_truoc"]:
    print("CẢNH BÁO: cài GPT-QModel đã đổi torch. Restart kernel và báo lại trước khi chạy tiếp.")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_JSONL = os.path.join(OUT_DIR, "kiem_gptqmodel.jsonl"); open(OUT_JSONL, "w").close()
def ghi(rec):
    with open(OUT_JSONL, "a") as f: f.write(json.dumps(rec, ensure_ascii=False) + "\n")
ghi({"buoc": "phien_ban", **INFO})

{'torch_truoc': '2.15.0+cu132', 'torch_sau': '2.15.0+cu132', 'transformers': '5.22.2', 'numpy': '2.2.6', 'gptqmodel': '7.6.0', 'gpu': 'CPU'}


## Ô 2. Tên đầu ra (lm head) của GPT-NeoX trong transformers đang dùng

transformers 5.22.2 đặt tên `lm_head`; định nghĩa `gpt_neox` của GPT-QModel (7.6.0 và bản mới nhất 13/11) ghi `embed_out`.
Khi Claude chạy thử, lệch tên này làm `save()` lỗi nếu bật lưu tạm ra đĩa (`offload_to_disk`). Notebook tắt `offload_to_disk`.

In [4]:
from transformers import GPTNeoXConfig, GPTNeoXForCausalLM
_m = GPTNeoXForCausalLM(GPTNeoXConfig(vocab_size=64, hidden_size=16, num_hidden_layers=1, num_attention_heads=2, intermediate_size=32))
head_names = [n for n, _ in _m.named_parameters() if "layers" not in n]
del _m
from gptqmodel.models.definitions.gpt_neox import GPTNeoXQModel
rec = {"buoc": "ten_lm_head", "transformers_params": head_names, "gptqmodel_lm_head": GPTNeoXQModel.lm_head}
print(rec); ghi(rec)

/home/claude/venv_gq/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1113 08:47:37.591000 3323 dist-packages/torch/utils/_pytree.py:646] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


W1113 08:47:37.620000 3323 dist-packages/torch/utils/_pytree.py:646] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


INFO  LogBar: headless/CI mode; progress animations disabled.                   

WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.

INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.

INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           

INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.6.0
Transformers : 5.22.2
Torch        : 2.15.0+cu132
Triton       : 3.9.0

{'buoc': 'ten_lm_head', 'transformers_params': ['gpt_neox.embed_in.weight', 'gpt_neox.final_layer_norm.weight', 'gpt_neox.final_layer_norm.bias', 'lm_head.weight'], 'gptqmodel_lm_head': 'embed_out'}


## Ô 3. Nạp model FP32 và dữ liệu hiệu chỉnh

Thật: Pythia-70M fp32 (D-04); hiệu chỉnh = `N_CALIB` đoạn `CALIB_LEN` token lấy từ WikiText-2 train (giống mặc định của Ouyang et al.: wikitext2 [GỢI Ý, chưa kiểm số đoạn và độ dài của họ]).
TOY: GPT-NeoX ngẫu nhiên cỡ Pythia-70M, câu tự viết.

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, PreTrainedTokenizerFast
torch.manual_seed(0)
dev = "cuda" if torch.cuda.is_available() else "cpu"
if TOY_TEST:
    tok = PreTrainedTokenizerFast(tokenizer_file=TOY_TOKENIZER_JSON, eos_token="<|endoftext|>", bos_token="<|endoftext|>", unk_token="<|endoftext|>")
    cfg = GPTNeoXConfig(vocab_size=50304, hidden_size=512, num_hidden_layers=6, num_attention_heads=8, intermediate_size=2048,
                        rotary_pct=0.25, max_position_embeddings=2048, use_parallel_residual=True, tie_word_embeddings=False)
    fp = GPTNeoXForCausalLM(cfg).eval()
    words = "the managers that injured the pilot are no consultant that has helped the taxi driver has shown any what the young man did was make".split()
    g = torch.Generator().manual_seed(0)
    texts = [" ".join(words[i] for i in torch.randint(len(words), (CALIB_LEN // 2,), generator=g).tolist()) for _ in range(N_CALIB + N_PPL)]
    calib_texts, ppl_texts = texts[:N_CALIB], texts[N_CALIB:]
else:
    tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
    fp = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, torch_dtype=torch.float32).eval()
    from datasets import load_dataset
    wt_train = "\n\n".join(load_dataset("wikitext", "wikitext-2-raw-v1", split="train")["text"])
    wt_test = "\n\n".join(load_dataset("wikitext", "wikitext-2-raw-v1", split="test")["text"])
    ids = tok(wt_train, return_tensors="pt")["input_ids"][0]
    g = torch.Generator().manual_seed(0)
    starts = torch.randint(0, len(ids) - CALIB_LEN, (N_CALIB,), generator=g).tolist()
    calib_texts = [tok.decode(ids[s:s + CALIB_LEN]) for s in starts]
    ppl_texts = None
tok.pad_token = tok.eos_token
FP_DIR = os.path.join(OUT_DIR, "fp_model"); fp.save_pretrained(FP_DIR); tok.save_pretrained(FP_DIR)
calib = []
for t in calib_texts:
    e = tok(t, truncation=True, max_length=CALIB_LEN)
    calib.append({"input_ids": e["input_ids"], "attention_mask": e["attention_mask"]})
print("calib:", len(calib), "đoạn, độ dài TB", sum(len(c["input_ids"]) for c in calib) / len(calib))
W_FP = {n: p.detach().clone() for n, p in fp.named_parameters() if p.ndim == 2 and ".layers." in n}
print("ma trận trong các khối:", len(W_FP))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.23it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.21it/s]

calib: 32 đoạn, độ dài TB 64.28125
ma trận trong các khối: 24


## Ô 4. RTN cùng cấu hình (mốc so sánh)

Bất đối xứng theo nhóm `GROUP_SIZE` cột đầu vào, áp lên 4 ma trận Linear mỗi khối, giống mô tả G0b (sổ §9).
**Khi gộp vào notebook chính, thay bằng hàm RTN trong `notebooks/00_g0_pilot.ipynb` (QT-25)**; hàm ở đây chỉ để so.

In [6]:
def rtn(w, bits, gs, sym):
    out = torch.empty_like(w); qmax = 2 ** bits - 1
    for g0 in range(0, w.shape[1], gs):
        x = w[:, g0:g0 + gs]
        if sym:
            s = (x.abs().amax(1, keepdim=True) / (2 ** (bits - 1) - 1)).clamp(min=1e-8)
            out[:, g0:g0 + gs] = (x / s).round().clamp(-(2 ** (bits - 1)), 2 ** (bits - 1) - 1) * s
        else:
            mn, mx = x.amin(1, keepdim=True), x.amax(1, keepdim=True)
            s = ((mx - mn) / qmax).clamp(min=1e-8); z = (-mn / s).round()
            out[:, g0:g0 + gs] = ((x / s).round() + z).clamp(0, qmax).sub(z) * s
    return out
W_RTN = {n: rtn(w, BITS, GROUP_SIZE, SYM) for n, w in W_FP.items()}
print("RTN xong:", len(W_RTN))

RTN xong: 24


## Ô 5. GPTQ bằng GPT-QModel, lưu, nạp lại bằng backend thuần torch

Ghi mọi tham số thật sự được dùng (kể cả giá trị mặc định do thư viện tự điền).

In [7]:
from gptqmodel import GPTQModel, QuantizeConfig, BACKEND
qcfg = QuantizeConfig(bits=BITS, group_size=GROUP_SIZE, sym=SYM, desc_act=DESC_ACT, act_group_aware=ACT_GROUP_AWARE,
                      damp_percent=DAMP_PERCENT, offload_to_disk=False)
used = {k: (str(getattr(qcfg, k)) if getattr(qcfg, k, None) is not None else None)
        for k in ["bits", "group_size", "sym", "desc_act", "act_group_aware", "damp_percent", "true_sequential", "static_groups", "mse", "format"]}
print(used)
t0 = time.time()
qm = GPTQModel.load(FP_DIR, qcfg)
qm.quantize(calib, batch_size=4)
Q_DIR = os.path.join(OUT_DIR, "gptq_model"); qm.save(Q_DIR)
t_quant = time.time() - t0
del qm
qm2 = GPTQModel.load(Q_DIR, backend=BACKEND.TORCH, device="cpu")
QMODS = {n.split("gpt_neox.", 1)[-1]: m for n, m in qm2.model.named_modules() if hasattr(m, "dequantize_weight")}
rec = {"buoc": "gptq", "cau_hinh_thuc_te": used, "giay_luong_tu_hoa": round(t_quant, 1), "so_module": len(QMODS)}
print(rec); ghi(rec)

{'bits': '4', 'group_size': '128', 'sym': 'False', 'desc_act': 'False', 'act_group_aware': 'False', 'damp_percent': '0.05', 'true_sequential': 'True', 'static_groups': 'False', 'mse': '0.0', 'format': 'FORMAT.GPTQ'}
INFO  Loader: Auto dtype (CPU + Torch Fused): `torch.bfloat16`                  

INFO:tokenicer.tokenicer:Tokenicer: Auto fixed pad_token_id=0 (token='<|endoftext|>').


INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]

INFO  Loader: loading model directly to CPU (not using meta device or turtle_model)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 1201.33it/s]

... (278 outputs left out)

INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.

INFO  Model: Updated `generation_config`: GenerationConfig {
  "bos_token_id": 0,
  "do_sample": true,
  "eos_token_id": 2,
  "output_attentions": false,
  "output_hidden_states": false,
  "pad_token_id": 0,
  "use_cache": true
}


INFO  Kernel: loaded -> `[TorchLinear]`                                         

{'buoc': 'gptq', 'cau_hinh_thuc_te': {'bits': '4', 'group_size': '128', 'sym': 'False', 'desc_act': 'False', 'act_group_aware': 'False', 'damp_percent': '0.05', 'true_sequential': 'True', 'static_groups': 'False', 'mse': '0.0', 'format': 'FORMAT.GPTQ'}, 'giay_luong_tu_hoa': 10.1, 'so_module': 24}


## Ô 6. Lấy Q(W) ở fp32 và kiểm

Không dùng `dequantize_model()` của GPT-QModel: hàm này ép trọng số về **float16** (đọc trong mã nguồn 7.6.0, `nn_modules/qlinear/torch.py`).
Gọi `dequantize_weight()` từng module rồi đổi sang fp32. Kiểm: mỗi nhóm có không quá 2^BITS giá trị khác nhau; so sai số tương đối với RTN; ghi kiểu dữ liệu của `scales`.

In [8]:
W_GPTQ = {}
for n, w in W_FP.items():
    key = n.split("gpt_neox.", 1)[-1].rsplit(".weight", 1)[0]
    m = QMODS[key]
    wq = m.dequantize_weight().T.to(torch.float32).cpu()
    assert wq.shape == w.shape, (n, wq.shape, w.shape)
    W_GPTQ[n] = wq
    rows = range(0, w.shape[0], max(1, w.shape[0] // 16))
    nuniq = max(len(torch.unique(wq[r, g0:g0 + GROUP_SIZE])) for r in rows for g0 in range(0, w.shape[1], GROUP_SIZE))
    rec = {"buoc": "module", "ten": n, "shape": list(w.shape), "toi_da_gia_tri_moi_nhom": nuniq,
           "sai_so_tuong_doi_gptq": round(float((wq - w).norm() / w.norm()), 4),
           "sai_so_tuong_doi_rtn": round(float((W_RTN[n] - w).norm() / w.norm()), 4),
           "scales_dtype": str(m.scales.dtype)}
    ghi(rec)
    if n.endswith("layers.0.attention.query_key_value.weight") or n.endswith("mlp.dense_4h_to_h.weight"):
        print(rec)
import statistics
tong = {"buoc": "tong_sai_so", "gptq_trung_vi": statistics.median(float((W_GPTQ[n] - W_FP[n]).norm() / W_FP[n].norm()) for n in W_FP),
        "rtn_trung_vi": statistics.median(float((W_RTN[n] - W_FP[n]).norm() / W_FP[n].norm()) for n in W_FP)}
print(tong); ghi(tong)

{'buoc': 'module', 'ten': 'gpt_neox.layers.0.attention.query_key_value.weight', 'shape': [1536, 512], 'toi_da_gia_tri_moi_nhom': 16, 'sai_so_tuong_doi_gptq': 0.1128, 'sai_so_tuong_doi_rtn': 0.1004, 'scales_dtype': 'torch.bfloat16'}


{'buoc': 'module', 'ten': 'gpt_neox.layers.0.mlp.dense_4h_to_h.weight', 'shape': [512, 2048], 'toi_da_gia_tri_moi_nhom': 16, 'sai_so_tuong_doi_gptq': 0.1051, 'sai_so_tuong_doi_rtn': 0.1006, 'scales_dtype': 'torch.bfloat16'}
{'buoc': 'module', 'ten': 'gpt_neox.layers.1.mlp.dense_4h_to_h.weight', 'shape': [512, 2048], 'toi_da_gia_tri_moi_nhom': 16, 'sai_so_tuong_doi_gptq': 0.1076, 'sai_so_tuong_doi_rtn': 0.1005, 'scales_dtype': 'torch.bfloat16'}
{'buoc': 'module', 'ten': 'gpt_neox.layers.2.mlp.dense_4h_to_h.weight', 'shape': [512, 2048], 'toi_da_gia_tri_moi_nhom': 16, 'sai_so_tuong_doi_gptq': 0.1092, 'sai_so_tuong_doi_rtn': 0.1005, 'scales_dtype': 'torch.bfloat16'}
{'buoc': 'module', 'ten': 'gpt_neox.layers.3.mlp.dense_4h_to_h.weight', 'shape': [512, 2048], 'toi_da_gia_tri_moi_nhom': 16, 'sai_so_tuong_doi_gptq': 0.1103, 'sai_so_tuong_doi_rtn': 0.1005, 'scales_dtype': 'torch.bfloat16'}
{'buoc': 'module', 'ten': 'gpt_neox.layers.4.mlp.dense_4h_to_h.weight', 'shape': [512, 2048], 'toi_da_gi

{'buoc': 'tong_sai_so', 'gptq_trung_vi': 0.11522268503904343, 'rtn_trung_vi': 0.10046404600143433}


## Ô 7. Perplexity: FP, RTN, GPTQ và một mức α

Thay trọng số vào chính model FP32 (giả lập: tính toán fp32, D-04). Nút vặn W(α) = W + α(Q(W) − W) (D-06) thử ở α = 0,5 với GPTQ.
Thật: `N_PPL` cửa sổ `PPL_LEN` token liên tiếp của WikiText-2 test. TOY: câu ngẫu nhiên.

In [9]:
@torch.no_grad()
def ppl(model, windows):
    model.to(dev).eval(); nll, n = 0.0, 0
    for w in windows:
        x = w.unsqueeze(0).to(dev)
        out = model(input_ids=x, labels=x)
        nll += out.loss.item() * (x.shape[1] - 1); n += x.shape[1] - 1
    return float(torch.exp(torch.tensor(nll / n)))

if TOY_TEST:
    windows = [torch.tensor(tok(t)["input_ids"][:PPL_LEN]) for t in ppl_texts]
else:
    tid = tok(wt_test, return_tensors="pt")["input_ids"][0]
    windows = [tid[i * PPL_LEN:(i + 1) * PPL_LEN] for i in range(N_PPL)]

def gan(model, Wdict):
    sd = model.state_dict()
    for n, w in Wdict.items(): sd[n].copy_(w)
    return model

res = {"buoc": "ppl", "fp": ppl(fp, windows)}
for ten, Wd in [("rtn", W_RTN), ("gptq", W_GPTQ),
                ("gptq_alpha_0.5", {n: W_FP[n] + 0.5 * (W_GPTQ[n] - W_FP[n]) for n in W_FP})]:
    gan(fp, Wd); res[ten] = ppl(fp, windows)
gan(fp, W_FP)
res = {k: (round(v, 3) if isinstance(v, float) else v) for k, v in res.items()}
print(res); ghi(res)

{'buoc': 'ppl', 'fp': 58563.961, 'rtn': 57608.906, 'gptq': 58608.992, 'gptq_alpha_0.5': 58590.102}


## Ô 8. Tóm tắt để mang về phiên chat

Mang về: file `kiem_gptqmodel.jsonl` và ảnh chụp ô 1 (phiên bản). Không cần mang thư mục model.

In [10]:
for line in open(OUT_JSONL):
    r = json.loads(line)
    if r["buoc"] != "module": print(line.rstrip())

{"buoc": "phien_ban", "torch_truoc": "2.15.0+cu132", "torch_sau": "2.15.0+cu132", "transformers": "5.22.2", "numpy": "2.2.6", "gptqmodel": "7.6.0", "gpu": "CPU"}
{"buoc": "ten_lm_head", "transformers_params": ["gpt_neox.embed_in.weight", "gpt_neox.final_layer_norm.weight", "gpt_neox.final_layer_norm.bias", "lm_head.weight"], "gptqmodel_lm_head": "embed_out"}
{"buoc": "gptq", "cau_hinh_thuc_te": {"bits": "4", "group_size": "128", "sym": "False", "desc_act": "False", "act_group_aware": "False", "damp_percent": "0.05", "true_sequential": "True", "static_groups": "False", "mse": "0.0", "format": "FORMAT.GPTQ"}, "giay_luong_tu_hoa": 10.1, "so_module": 24}
{"buoc": "tong_sai_so", "gptq_trung_vi": 0.11522268503904343, "rtn_trung_vi": 0.10046404600143433}
{"buoc": "ppl", "fp": 58563.961, "rtn": 57608.906, "gptq": 58608.992, "gptq_alpha_0.5": 58590.102}
